# 9.1 Behavioral adjustment factors

## Introduction 

### Structural noise reduction approach

**Can behavioral segmentation reduce noise from search-demand data?**
Previous analyses revealed that search-demand relationships are highly heterogeneous across tourism destinations and historical periods.
One possible explanation is that highly aggregated tourism demand datasets combine structurally different travel behaviors with fundamentally distinct digital information-seeking intensities.

For example:

- leisure tourism generally generates stronger pre-travel search behavior,
- VFR travel (visiting friends or relatives), short-distance trips, or corporate travel may produce substantially weaker digital search signals.

As a result, aggregated tourism demand datasets may dilute or distort the observable relationship between search behavior and tourism demand.

This notebook explores whether search-demand coherence improves after introducing behavioral adjustment factors derived from external tourism segmentation statistics published by the Spanish National Statistics Institute (INE) through the:

Encuesta de Turismo de Residentes (ETR).
Methodology source:
https://www.ine.es/daco/daco42/etr/etr_metodologia.pdf

The objective is not to reconstruct exact country-level tourism segmentation, but rather to evaluate whether approximate behavioral weighting proxies can reduce structural noise and produce more coherent search-demand relationships.

In [109]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

## Load datasets

In [110]:
DATA_PATH = Path("../data/processed")
RAW_SURVEY_PATH = Path("../data/raw/ine_tourism_residents_survey/original_surveys")

RAW_SURVEY_PATH.mkdir(
    parents=True,
    exist_ok=True
)
    
ine_raw = pd.read_excel(
    RAW_SURVEY_PATH / "raw_INE_all_international_vs_continent_anual.xlsx",
    header=None,
    thousands="."
)

df_spain = pd.read_parquet(
    DATA_PATH /
    "07_master_demand.parquet"
)


c:\ProgramData\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning:

Workbook contains no default style, apply openpyxl's default



**Note on data suppression:** The INE ETR suppresses cells with insufficient sample size. África 2021 leisure trips are not available (NaN). For these cases, the national-level factor (`All`) is used as fallback in the merge step.

In [111]:


years = ine_raw.iloc[13, 1:8].astype(float).astype(int).tolist()

total   = ine_raw.iloc[14:19, [0,1,2,3,4,5,6,7]].copy()
leisure = ine_raw.iloc[14:19, [0,8,9,10,11,12,13,14]].copy()

total.columns   = ["continent_ine"] + years
leisure.columns = ["continent_ine"] + years

for y in years:
    total[y]   = pd.to_numeric(total[y],   errors="coerce")
    leisure[y] = pd.to_numeric(leisure[y], errors="coerce")

# Compute leisure rate per continent × year
df_factors = total[["continent_ine"]].copy()
for y in years:
    df_factors[y] = leisure[y] / total[y]

df_factors_long = df_factors.melt(
    id_vars="continent_ine",
    var_name="year",
    value_name="behavioral_factor"
)
df_factors_long["year"] = df_factors_long["year"].astype(int)

# Continent mapping: INE names → df_non_europe names
continent_map = {
    "América":              "America",
    "África":               "Africa",
    "Resto del Mundo":      "Asia",
    "Europa (sin España)":  "Europe",
    "Extranjera":           "All",
}
df_factors_long["continent"] = df_factors_long["continent_ine"].map(continent_map)
df_factors_long = df_factors_long.dropna(subset=["behavioral_factor", "continent"])

print(df_factors_long.pivot(index="continent", columns="year", values="behavioral_factor").round(3))


year       2019  2020  2021  2022  2023  2024  2025
continent                                          
Africa    0.383 0.331   NaN 0.326 0.354 0.304 0.403
All       0.621 0.540 0.529 0.610 0.620 0.608 0.612
America   0.473 0.360 0.302 0.449 0.414 0.380 0.382
Asia      0.697 0.582 0.575 0.687 0.659 0.614 0.664
Europe    0.652 0.568 0.570 0.654 0.676 0.673 0.658


## Merge

In [112]:
# Merge by continent × year instead of national factor
df_spain = pd.read_parquet(DATA_PATH / "07_master_demand.parquet")

df_ine_adjusted = df_spain.merge(
    df_factors_long[["continent", "year", "behavioral_factor"]],
    on=["continent", "year"],
    how="left"
)

# Fill missing with national average (Extranjera)
national_factor = df_factors_long[df_factors_long["continent"] == "All"].set_index("year")["behavioral_factor"]
df_ine_adjusted["behavioral_factor"] = df_ine_adjusted.apply(
    lambda row: row["behavioral_factor"] if pd.notna(row["behavioral_factor"])
    else national_factor.get(row["year"], np.nan),
    axis=1
)

display(df_ine_adjusted[["destination", "continent", "year", "behavioral_factor"]].head(20))


,destination,continent,year,behavioral_factor
0,Albania,Europa,2019,0.621
1,Alemania,Europa,2019,0.621
2,Andorra,Europa,2019,0.621
3,Angola,Africa,2019,0.383
4,Arabia Saudí,Asia,2019,0.697
5,Argelia,Africa,2019,0.383
6,Argentina,America,2019,0.473
7,Armenia,Asia,2019,0.697
8,Australia,Oceania,2019,0.621
9,Austria,Europa,2019,0.621


## Total tourists adjusted

$$ AdjustedDemand = TotalDemand \times BehavioralScore $$

In [113]:
df_ine_adjusted["adjusted_tourists"] = (
    df_ine_adjusted["total_tourists"]
    *
    df_ine_adjusted["behavioral_factor"]
)

display(df_ine_adjusted.head())

,period,season,month,year,destination,destination_clean,continent,eu,mediterranean,flight_dependency,...,destination_segment,include_in_analysis,covid_period,total_tourists,trend_index,monthly_searches,competition,cpc,behavioral_factor,adjusted_tourists
0,2019-07-01,summer,7,2019,Albania,albania,Europa,True,True,medium,...,short_haul,False,pre_covid,264,0.000,"1,900.000",0.690,0.660,0.621,164.014
1,2019-07-01,summer,7,2019,Alemania,alemania,Europa,True,False,medium,...,short_haul,False,pre_covid,101502,26.500,"1,900.000",0.520,0.470,0.621,"63,059.598"
2,2019-07-01,summer,7,2019,Andorra,andorra,Europa,True,False,medium,...,short_haul,False,pre_covid,58532,17.500,"1,900.000",0.630,0.600,0.621,"36,363.859"
3,2019-07-01,summer,7,2019,Angola,angola,Africa,False,False,high,...,long_haul,True,pre_covid,63,0.000,90.000,0.360,1.140,0.383,24.160
4,2019-07-01,summer,7,2019,Arabia Saudí,arabia saudi,Asia,False,False,high,...,long_haul,True,pre_covid,216,0.000,NaN,NaN,NaN,0.697,150.576


## Original vs adjusted correlation comparison

In [114]:
df_commercial_adjusted = (
    df_ine_adjusted[
        df_ine_adjusted["agency_profile"]
        .isin(["medium", "high"])
    ]
    .copy()
)

## Recalculate correlations

Between total_tourists and digital demand-signals

### 1. Country-level correlations

## Destination-level comparison

In [115]:
#correlation with original total tourists

country_corr_original = (
    df_commercial_adjusted
    .groupby("destination")
    .apply(
        lambda x: x["monthly_searches"].corr(x["total_tourists"]),
        include_groups=False
    )
    .reset_index(name="original_correlation")
)
 
country_corr_adjusted = (
    df_commercial_adjusted
    .groupby("destination")
    .apply(
        lambda x: x["monthly_searches"].corr(x["adjusted_tourists"]),
        include_groups=False
    )
    .reset_index(name="adjusted_correlation")
)
 
corr_comparison = country_corr_original.merge(
    country_corr_adjusted, on="destination"
)
corr_comparison["improvement"] = (
    corr_comparison["adjusted_correlation"]
    - corr_comparison["original_correlation"]
)
print("The 10 destinations where it improves the most:") 
display(corr_comparison.sort_values("improvement", ascending=False).head(10))
print("The 10 destinations where correlation worsens the most:") 
display(corr_comparison.sort_values("improvement").head(10))

The 10 destinations where it improves the most:


,destination,original_correlation,adjusted_correlation,improvement
43,Montenegro,0.253,0.403,0.150
5,Brasil,0.649,0.794,0.145
10,Chile,0.247,0.327,0.081
36,Letonia,0.187,0.265,0.078
32,Japón,0.621,0.686,0.066
9,Canadá,0.362,0.423,0.061
50,Panamá,0.367,0.416,0.049
20,Eslovenia,0.398,0.440,0.042
2,Armenia,0.126,0.166,0.040
68,Turquía,0.475,0.514,0.040


The 10 destinations where correlation worsens the most:


,destination,original_correlation,adjusted_correlation,improvement
71,Uganda,-0.194,-0.349,-0.154
59,Senegal,0.313,0.205,-0.108
48,Omán,0.307,0.234,-0.073
70,Ucrania,0.046,-0.006,-0.052
56,República Dominicana,0.530,0.490,-0.040
67,Tanzania,0.401,0.364,-0.038
63,Sudáfrica,0.414,0.383,-0.031
7,Cabo Verde,0.376,0.348,-0.028
18,Egipto,0.773,0.747,-0.026
40,Maldivas,0.528,0.505,-0.023


In [116]:

destination_summary_adjusted = (
    df_commercial_adjusted
    .groupby("destination")
    .agg({
        "monthly_searches":"mean",
        "adjusted_tourists":"mean"
    })
    .reset_index()
)



destination_summary_original = (
    df_commercial_adjusted
    .groupby("destination")
    .agg({
        "monthly_searches":"mean",
        "total_tourists":"mean"
    })
    .reset_index()
)


## Validation summary

In [117]:

summary_comparison = pd.DataFrame({
    "metric": [
        "mean",
        "median",
        "std",
        "positive_pct"
    ],

    "original_search": [
        corr_summary["original_monthly_corr"].mean(),
        corr_summary["original_monthly_corr"].median(),
        corr_summary["original_monthly_corr"].std(),
        (
            corr_summary["original_monthly_corr"] > 0
        ).mean()
    ],

    "adjusted_search": [
        corr_summary["adjusted_search_corr"].mean(),
        corr_summary["adjusted_search_corr"].median(),
        corr_summary["adjusted_search_corr"].std(),
        (
            corr_summary["adjusted_search_corr"] > 0
        ).mean()
    ],

    "original_trends": [
        corr_summary["original_trend_corr"].mean(),
        corr_summary["original_trend_corr"].median(),
        corr_summary["original_trend_corr"].std(),
        (
            corr_summary["original_trend_corr"] > 0
        ).mean()
    ],

    "adjusted_trends": [
        corr_summary["adjusted_trend_corr"].mean(),
        corr_summary["adjusted_trend_corr"].median(),
        corr_summary["adjusted_trend_corr"].std(),
        (
            corr_summary["adjusted_trend_corr"] > 0
        ).mean()
    ]
})

display(
    summary_comparison.round(3)
)


,metric,original_search,adjusted_search,original_trends,adjusted_trends
0,mean,0.271,0.273,0.271,0.287
1,median,0.293,0.306,0.265,0.308
2,std,0.272,0.283,0.270,0.279
3,positive_pct,0.876,0.858,0.823,0.823


# Interpretation and conclusions

## Does a continent-level behavioral adjustment improve search-demand correlations?

The continent-level behavioral adjustment (leisure rate per continent × year,
derived from INE ETR 2019–2025) was applied to `total_tourists` to reduce
structural noise from non-leisure travel (VFR, business, student mobility).

### Results

| Metric | Original | National factor | Continent factor |
|--------|----------|-----------------|------------------|
| Mean (searches) | 0.272 | 0.136 | 0.273 |
| Median (searches) | 0.298 | 0.180 | 0.306 |
| % positive | 87.5% | 70.5% | 85.8% |

### Interpretation

The continent-level factor produces marginal but consistent improvements over the unadjusted baseline, unlike the national factor which severely
degrades correlations.

This suggests that continent-level leisure segmentation captures meaningful structural differences — especially América at ~38% leisure vs Asia at ~61%
— while a single national average masks them.

The improvement remains modest because the INE ETR continental aggregation still masks significant within-continent heterogeneity: Colombia and Argentina
both receive the same 38% América factor despite different VFR profiles. Similarly, Japan and UAE receive the same Asia factor despite very different travel motivations.

### Conclusion

> **The continent-level behavioral factor is the recommended adjustment.**
> It produces marginally better correlations than the unadjusted baseline and is conceptually sounder than a national average. The unadjusted signal remains a valid fallback given the modest improvement.

### Future work

A country-level leisure rate would be the ideal improvement, but is not available in the INE ETR (which only disaggregates by continent).
A potential future approach would combine INE ETR continent factors with UNWTO or NTTO country-level VFR estimates to build destination-specific
adjustment factors.